In [16]:
# %pip install optuna 

import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from importlib import reload
import src.utils.purged_cv 
reload(src.utils.purged_cv )
from src.utils.purged_cv import PurgedTimeSeriesSplit
from sklearn.metrics import ndcg_score

In [6]:
df = pd.read_parquet('../data/processed/run_20260215_112600/modeling_dataset.parquet')

In [7]:
fred_features = [
    'sector', 'DFF', 'DGS10', 'DGS2', 'T10Y2Y', 'CPIAUCSL',
    'CPILFESL', 'PCEPI', 'GDP', 'GDPC1', 'INDPRO', 'UNRATE', 'PAYEMS',
    'ICSA', 'UMCSENT', 'RSXFS', 'VIXCLS', 'DCOILWTICO', 'M2SL', 'TOTALSL'
]

rank_features = [col for col in df.columns if "rank" in col]

zscore_features = [col for col in df.columns if col.endswith('_zscore')]

quintile_features = [col for col in df.columns if col.endswith('_quintile')]

modeling_features = fred_features + rank_features + zscore_features + quintile_features

In [21]:
# Convert categorical columns to category dtype
categorical_features = ['sector']
for col in categorical_features:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Time-based split (1-month embargo for test)
train = df[df['date'] < '2022-05-01']
test = df[df['date'] >= '2022-06-01']

train = train[train["return_t+10"].notna()]
test = test[test["return_t+10"].notna()]

In [29]:
# ==========================================================
# Helper: Compute NDCG per date (ranking group)
# ==========================================================

def compute_mean_ndcg(df, k=20):
    """
    Computes mean NDCG@k across all dates.
    Assumes df contains:
        - 'date'
        - 'return_t+10'
        - 'pred'
    """
    ndcgs = []

    for _, group in df.groupby("date"):
        if len(group) < 2:
            continue

        y_true = group["return_t+10"].values.reshape(1, -1)
        y_pred = group["pred"].values.reshape(1, -1)

        score = ndcg_score(y_true, y_pred, k=k)
        ndcgs.append(score)

    return np.mean(ndcgs)


def compute_ic(df):

    ics = []

    for _, g in df.groupby("date"):
        if len(g) < 10:
            continue

        ic = g["pred"].corr(g["excess_return_t+10"], method="spearman")
        ics.append(ic)

    return np.mean(ics)


# ==========================================================
# Bayesian Objective Function
# ==========================================================

def objective(trial):

    params = {
        "objective": "rank:pairwise",
        "eval_metric": "ndcg",
        "tree_method": "hist",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "max_depth": trial.suggest_int("max_depth", 1, 5),
        "min_child_weight": trial.suggest_float("min_child_weight", 1, 100),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "lambda": trial.suggest_float("lambda", 1e-3, 10, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "seed": 42
    }

    num_boost_round = 100

    ic_scores = []
    ndcg_scores = []

    splitter = PurgedTimeSeriesSplit(
        n_splits=3,
        purge_days=14,
        embargo_days=5
    )

    for fold_num, (train_idx, val_idx) in enumerate(
        splitter.split(train, train["date"])
    ):

        train_fold = train.loc[train_idx].sort_values("date")
        val_fold = train.loc[val_idx].sort_values("date")

        dtrain = xgb.DMatrix(
            train_fold[modeling_features],
            label=train_fold["return_t+10"],
            enable_categorical=True
        )
        dtrain.set_group(train_fold.groupby("date").size().values)

        dval = xgb.DMatrix(
            val_fold[modeling_features],
            label=val_fold["return_t+10"],
            enable_categorical=True
        )
        dval.set_group(val_fold.groupby("date").size().values)

        model = xgb.train(
            params,
            dtrain,
            num_boost_round=num_boost_round,
            # evals=[(dval, "validation")],
            # early_stopping_rounds=50,
            verbose_eval=False
        )

        val_fold = val_fold.copy()
        val_fold["pred"] = model.predict(dval)

        # --- Compute Metrics ---
        fold_ic = compute_ic(val_fold)
        fold_ndcg = compute_mean_ndcg(val_fold, k=20)

        ic_scores.append(fold_ic)
        ndcg_scores.append(fold_ndcg)

        # Optional: Report intermediate IC for pruning
        trial.report(np.mean(ic_scores), step=fold_num)
        if trial.should_prune():
            raise optuna.TrialPruned()

    mean_ic = np.mean(ic_scores)
    mean_ndcg = np.mean(ndcg_scores)

    # Store secondary metric
    trial.set_user_attr("mean_ndcg", mean_ndcg)
    trial.set_user_attr("mean_ic", mean_ic)

    return mean_ic


## Run search

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(
    objective,
    n_trials=10,
    show_progress_bar=True
)

print("Best score:", study.best_value)
print("Best params:", study.best_params)

[I 2026-02-17 13:09:14,855] A new study created in memory with name: no-name-562c09fe-c6d4-4c22-9e16-42d117a16873
  0%|          | 0/10 [00:00<?, ?it/s]